In [2]:
from pathlib import Path
from tqdm.auto import tqdm
import joblib
import pandas as pd

In [4]:
EXECUTION_DATE = "20260708"
CV_LOSS_PATH : str = f'../../data/interim/cv/{EXECUTION_DATE}/loss/'

In [ ]:
files = sorted(Path(CV_LOSS_PATH).glob("*.jbl"))
files[0]

[PosixPath('../../data/interim/cv/20260708/loss/epanechnikov_adaptive___False___2___VAR___1.jbl')]

In [ ]:
files = sorted(Path(CV_LOSS_PATH).glob("*.jbl"))
# files = [sorted(Path(CV_LOSS_PATH).glob("*.jbl"))[10]]

df_loss_base_long = pd.concat(
    (joblib.load(f) for f in files),
    ignore_index=True,
)

joblib.dump(
    df_loss_base_long,
    f"{CV_MAIN_PATH}cv_results_cross_comparison.jbl"
)

In [17]:
df_loss_base_long.kde_model.unique()

<ArrowStringArray>
[                    'gaussian_rot_robust___False',
                 'gaussian_rot_robust___expanding',
                'gaussian_rot_robust___rolling_w5',
               'gaussian_rot_robust___rolling_w21',
               'gaussian_rot_robust___rolling_w63',
                            'gaussian_rot___False',
                        'gaussian_rot___expanding',
                       'gaussian_rot___rolling_w5',
                      'gaussian_rot___rolling_w21',
                      'gaussian_rot___rolling_w63',
                 'epanechnikov_rot_robust___False',
             'epanechnikov_rot_robust___expanding',
            'epanechnikov_rot_robust___rolling_w5',
           'epanechnikov_rot_robust___rolling_w21',
           'epanechnikov_rot_robust___rolling_w63',
                        'epanechnikov_rot___False',
                    'epanechnikov_rot___expanding',
                   'epanechnikov_rot___rolling_w5',
                  'epanechnikov_rot___rolling

In [18]:
df_loss_base_long[df_loss_base_long["kde_model"]=="gaussian_rot_robust___False"]

,kde_model,forecast_model,kde_params,postprocessing,dFPC_dimensions,etahat_fc_model,etahat_fc_model_spec,etahat_fc_model_lags,date,KLD,JSD,L1_norm,L2_norm,LINF_norm
0,gaussian_rot_robust___False,epanechnikov_adaptive___False___2___VAR___1,epanechnikov_adaptive,False,2,VAR,VAR,1,2025-05-05,0.197006,0.021316,28281.196614,1296.860634,116.901271
1,gaussian_rot_robust___False,epanechnikov_adaptive___False___2___VAR___1,epanechnikov_adaptive,False,2,VAR,VAR,1,2025-05-06,0.107903,0.005431,20557.548723,1023.091247,104.801512
2,gaussian_rot_robust___False,epanechnikov_adaptive___False___2___VAR___1,epanechnikov_adaptive,False,2,VAR,VAR,1,2025-05-07,0.234578,0.006131,19591.873370,828.184520,66.635835
3,gaussian_rot_robust___False,epanechnikov_adaptive___False___2___VAR___1,epanechnikov_adaptive,False,2,VAR,VAR,1,2025-05-08,0.446441,0.024658,41908.332841,1958.401612,202.167944
4,gaussian_rot_robust___False,epanechnikov_adaptive___False___2___VAR___1,epanechnikov_adaptive,False,2,VAR,VAR,1,2025-05-09,0.542028,0.025805,43037.251622,2648.964087,380.912866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1284736,gaussian_rot_robust___False,epanechnikov_adaptive___rolling_w63___2___regr...,epanechnikov_adaptive,rolling_w63,2,regression,adalasso,1,2025-08-21,0.764101,0.027407,43917.400251,2504.472797,337.985295
1284737,gaussian_rot_robust___False,epanechnikov_adaptive___rolling_w63___2___regr...,epanechnikov_adaptive,rolling_w63,2,regression,adalasso,1,2025-08-22,0.854375,0.059444,61741.778654,3587.280896,450.684124
1284738,gaussian_rot_robust___False,epanechnikov_adaptive___rolling_w63___2___regr...,epanechnikov_adaptive,rolling_w63,2,regression,adalasso,1,2025-08-25,0.108783,0.015851,28110.046396,1725.495531,220.485894
1284739,gaussian_rot_robust___False,epanechnikov_adaptive___rolling_w63___2___regr...,epanechnikov_adaptive,rolling_w63,2,regression,adalasso,1,2025-08-26,0.159933,0.009542,21242.724572,1275.499488,159.261930


In [30]:
from pathlib import Path
import joblib
import pandas as pd

metrics = ["KLD", "JSD", "L1_norm", "L2_norm", "LINF_norm"]

group_cols = [
    "forecast_model",
    "postprocessing",
    "dFPC_dimensions",
    "etahat_fc_model",
    "etahat_fc_model_spec",
    "etahat_fc_model_lags",
]

records = []

for file in sorted(Path(CV_LOSS_PATH).glob("*.jbl")):

    df = joblib.load(file)

    summary = (
        df
        .melt(
            id_vars=group_cols,
            value_vars=metrics,
            var_name="metric",
            value_name="value",
        )
        .groupby(group_cols + ["metric"])["value"]
        .agg(
            mean="mean",
            std="std",
            median="median",
            q25=lambda x: x.quantile(.25),
            q75=lambda x: x.quantile(.75),
            iqr=lambda x: x.quantile(.75) - x.quantile(.25),
            minimum="min",
            maximum="max",
            n="count",
        )
        .reset_index()
    )

    records.append(summary)

df_summary = (
    pd.concat(records, ignore_index=True)
      .sort_values(["metric", "mean"])
      .reset_index(drop=True)
)

df_summary.head()

,forecast_model,postprocessing,dFPC_dimensions,etahat_fc_model,etahat_fc_model_spec,etahat_fc_model_lags,metric,mean,std,median,q25,q75,iqr,minimum,maximum,n
0,epanechnikov_adaptive___rolling_w5___5___regre...,rolling_w5,5,regression,randomforest,2,JSD,0.078892,0.073348,0.059370,0.015057,0.119009,0.103953,0.000454,0.338286,5265
1,epanechnikov_adaptive___rolling_w5___6___regre...,rolling_w5,6,regression,randomforest,2,JSD,0.078944,0.073424,0.059421,0.015038,0.119049,0.104010,0.000429,0.339513,5265
2,epanechnikov_adaptive___rolling_w5___4___regre...,rolling_w5,4,regression,randomforest,2,JSD,0.078950,0.073177,0.059655,0.015175,0.119143,0.103968,0.000514,0.338092,5265
3,epanechnikov_adaptive___rolling_w5___3___regre...,rolling_w5,3,regression,randomforest,2,JSD,0.079113,0.073346,0.059738,0.015189,0.119378,0.104190,0.000508,0.338585,5265
4,epanechnikov_adaptive___rolling_w5___5___regre...,rolling_w5,5,regression,randomforest,3,JSD,0.079802,0.074367,0.060176,0.015251,0.120936,0.105686,0.000535,0.341934,5265


In [31]:
df_summary.query(
    "metric == 'KLD' and etahat_fc_model == 'VAR'"
).sort_values("mean")

,forecast_model,postprocessing,dFPC_dimensions,etahat_fc_model,etahat_fc_model_spec,etahat_fc_model_lags,metric,mean,std,median,q25,q75,iqr,minimum,maximum,n
711,epanechnikov_adaptive___rolling_w5___4___VAR___2,rolling_w5,4,VAR,VAR,2,KLD,0.852224,0.865871,0.605724,0.129494,1.264048,1.134554,0.004483,4.824669,5265
713,epanechnikov_adaptive___rolling_w5___5___VAR___2,rolling_w5,5,VAR,VAR,2,KLD,0.853267,0.867667,0.611310,0.128935,1.263418,1.134483,0.005502,5.000698,5265
717,epanechnikov_adaptive___rolling_w5___3___VAR___2,rolling_w5,3,VAR,VAR,2,KLD,0.854700,0.869958,0.606633,0.127176,1.266987,1.139811,0.002730,4.828839,5265
718,epanechnikov_adaptive___rolling_w5___4___VAR___3,rolling_w5,4,VAR,VAR,3,KLD,0.854981,0.871037,0.600957,0.131983,1.280299,1.148316,0.003746,4.741904,5265
719,epanechnikov_adaptive___rolling_w5___6___VAR___1,rolling_w5,6,VAR,VAR,1,KLD,0.855320,0.872341,0.601757,0.126586,1.269155,1.142569,0.004161,4.664884,5265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1333,epanechnikov_rot___False___3___VAR___3,False,3,VAR,VAR,3,KLD,2.534538,2.389478,1.708479,0.373996,4.309423,3.935427,0.011279,9.950714,5265
1336,epanechnikov_rot___False___4___VAR___4,False,4,VAR,VAR,4,KLD,2.549445,2.396087,1.709616,0.392312,4.304228,3.911916,0.012429,9.955109,5265
1340,epanechnikov_rot___False___3___VAR___4,False,3,VAR,VAR,4,KLD,2.551799,2.399631,1.719690,0.382977,4.321222,3.938245,0.012505,9.959175,5265
1341,epanechnikov_rot___False___5___VAR___4,False,5,VAR,VAR,4,KLD,2.566858,2.418409,1.730004,0.392614,4.344011,3.951397,0.012844,10.248544,5265


In [32]:
(
    df_summary.query("metric == 'KLD'")
    .groupby("postprocessing")["mean"]
    .mean()
    .sort_values()
)

postprocessing
rolling_w5     1.043921
rolling_w21    1.072535
rolling_w63    1.158141
expanding      1.197679
False          2.298073
Name: mean, dtype: float64

In [34]:
from IPython.display import display

config_cols = [
    "forecast_model",
    "postprocessing",
    "dFPC_dimensions",
    "etahat_fc_model",
    "etahat_fc_model_spec",
    "etahat_fc_model_lags",
]

for metric in df_summary["metric"].unique():

    print(f"\n{'='*90}")
    print(f"{metric}")
    print(f"{'='*90}")

    metric_df = df_summary.query("metric == @metric")

    for col in config_cols:

        result = (
            metric_df
            .groupby(col, dropna=False)["mean"]
            .mean()
            .sort_values()
            .rename("Mean loss")
            .reset_index()
        )

        if len(result) > 10:
            result = result.head(10)

        print(f"\n▶ {col}")

        display(
            result.style
                .format({"Mean loss": "{:.6f}"})
                .hide(axis="index")
                .background_gradient(
                    subset=["Mean loss"],
                    cmap="RdYlGn_r"      # green = better (lower)
                )
        )


JSD

▶ forecast_model


forecast_model,Mean loss
epanechnikov_adaptive___rolling_w5___5___regression_randomforest___2,0.078892
epanechnikov_adaptive___rolling_w5___6___regression_randomforest___2,0.078944
epanechnikov_adaptive___rolling_w5___4___regression_randomforest___2,0.078950
epanechnikov_adaptive___rolling_w5___3___regression_randomforest___2,0.079113
epanechnikov_adaptive___rolling_w5___5___regression_randomforest___3,0.079802
epanechnikov_adaptive___rolling_w5___4___regression_randomforest___3,0.079837
epanechnikov_adaptive___rolling_w5___6___regression_randomforest___3,0.079841
epanechnikov_adaptive___rolling_w5___3___regression_randomforest___3,0.079862
epanechnikov_adaptive___rolling_w5___2___regression_randomforest___2,0.080048
epanechnikov_adaptive___rolling_w5___5___regression_randomforest___1,0.080162



▶ postprocessing


postprocessing,Mean loss
rolling_w5,0.096601
rolling_w21,0.098412
rolling_w63,0.103557
expanding,0.104793
False,0.165566



▶ dFPC_dimensions


...

etahat_fc_model,Mean loss
VAR,835.444223
regression,837.155739



▶ etahat_fc_model_spec


etahat_fc_model_spec,Mean loss
randomforest,834.251775
VAR,835.444223
adalasso,840.047023



▶ etahat_fc_model_lags


etahat_fc_model_lags,Mean loss
2,835.498531
1,836.598886
3,836.656817
4,837.563404
